# Search the STAC catalog and read data

This example searches your Organization's STAC catalog with pystac-client, and reads one asset with rasterio.

You need `pystac-client`, `rasterio` and `matplotlib`. They are not dependencies of the SDK.

In [ ]:
import matplotlib.pyplot as plt
import rasterio
from pystac_client import Client
from rasterio.plot import show

from prescient_sdk import PrescientClient

## Open the catalog

`client.stac_url` is the STAC API of your Organization. `client.request_modifier` adds a current access token to each request that pystac-client sends.

In [ ]:
client = PrescientClient(env_file="config.env")

catalog = Client.open(client.stac_url, request_modifier=client.request_modifier)
catalog.title

## Search the catalog

This search gets the first item in the catalog. For other searches, see the [pystac-client documentation](https://pystac-client.readthedocs.io/en/stable/usage.html).

In [ ]:
results = catalog.search(max_items=1)
items = list(results.items())

## Read an asset with rasterio

Asset links point at the platform file proxy over HTTPS. To read an asset, send the access token in an `Authorization` header. You do not need AWS credentials.

`rasterio.Env(GDAL_HTTP_HEADERS=...)` gives GDAL the header. GDAL cannot get a new token during a read. The token is valid for a short time: 5 minutes on the platform today, and about 1 hour in a later platform release. When the token expires, make a new `rasterio.Env` with a new header from `client.headers`.

GDAL sends the header to every URL that you open in the `rasterio.Env` block. Open only Prescient asset links in that block.

In [ ]:
auth_header = f"Authorization: {client.headers['Authorization']}"

fig, ax = plt.subplots()

with rasterio.Env(GDAL_HTTP_HEADERS=auth_header):
    for item in items:
        # Use the first asset that has the "data" role.
        asset = next(a for a in item.assets.values() if "data" in (a.roles or []))
        with rasterio.open(asset.href) as src:
            show(src, ax=ax, cmap="gray")
            ax.set_axis_off()